## Mirror Mode Waves from Rosetta Data

### Functions

In [64]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                #my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
                
                
                df22=pd.DataFrame({'Beginning' : [array1[index1-(n/2)]], 'End' : [array1[index2+(n/2)]], 'Index1' : [index1], 'Index2': [index2]})
                my_df=pd.concat([my_df, df22],axis=0, ignore_index=True)
             
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            #my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
            
            df22=pd.DataFrame({'Beginning' : [array1[index1-(n/2)]], 'End' : [array1[index2+(n/2)]], 'Index1' : [index1], 'Index2': [index2]})
            my_df=pd.concat([my_df, df22],axis=0,ignore_index=True)
        
            
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

### Importing Data

#### Low Resolution Plasma Density

In [92]:
titles2=['time_epoch', 't_obt', 'usc','qv' ,'qf' ,'t_utc' ,'data source' ,'macroId' ,'npl','t']

#, sep='\s+'
#Import the plasma density dataframe
data_density=pd.read_table("C:/Users/atell/OneDrive/Escritorio/ESA/usc_v09+npl.txt", header=None, names=titles2,parse_dates=['t_utc'],low_memory=False)
data_density=data_density.iloc[np.arange(1,len(data_density)),:] #delete the first row
data_density=data_density.reset_index(drop=True)
#print(data_density)
data_plasma=data_density[[ 't_utc', 'npl']]

#print(data_plasma)

#### High Resolution Plasma Density

In [69]:
#INSERT THE HIGH RESOLUTION PLASMA FOLDER

#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #november 2014
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #december 2014

#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #february 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #march 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #april 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #may 2015

plasma_folder="C:/Users/atell/OneDrive/Escritorio/ESA/trainwave/LAP DATA" #trainwave 2015
#plasma_folder="C:/Users/atell/OneDrive/Escritorio/esa compuviejo/examplee2" #singleevent 2015

#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #july 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #august 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #september 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #october 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #november 2015
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #december 2015

#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #january 2016
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #february 2016
#plasma_folder="C:/Users/Ariel/Desktop/ESA/example2" #march 2016

#directory= os.scandir(plasma_folder)

#---------------------------------------------------------

list_of_files_plasma=get_directory(plasma_folder)

#List with the paths
newlist=[]
for item in list_of_files_plasma:
    newlist.append(plasma_folder+'/'+str(item))
print(newlist)  

year_plasma=[]
month_plasma=[]
day_plasma=[]
list_of_plasma=[] #List of arrays with plasma densities

#for i in np.arange(0, 2):
for i in np.arange(0, len(newlist)):
        title2=['t_utc','?','npl','??','???','????']
        path= str(newlist[i])
        data1= pd.read_table(path, header=None, names=title2, sep=',', engine='python', parse_dates=['t_utc'])
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        data2=data2.reset_index(drop=True)
        data2=data2[['t_utc','npl']]
        
        #--------------------------------------------------------------------------
        #Saving the dates
        path_time= pd.to_datetime(data2['t_utc']) #data2['t_utc']
        #It is needed to obtain the year, month and day of an specific path
        q=path_time.dt.year
        qq=path_time.dt.month
        qqq=path_time.dt.day
        year=q[0]
        month=qq[0]
        day=qqq[0]
        
        year_plasma.append(year)
        month_plasma.append(month)
        day_plasma.append(day)
        #------------------------------------------------------------------------------
        #Resample
        data2['index'] = pd.to_datetime(data2['t_utc'])
        data2.set_index('index', inplace=True)
        data2=data2.resample('1S').mean(numeric_only=True)

        data2['t_utc'] = pd.to_datetime(data2.index.values)
        data2= data2.reset_index()
        #time_lap=pd.to_datetime(data2['t_utc'])
        #---------------------------------------------------------------------------------

        #Filling the data gaps
        data2.t_utc = data2.t_utc.dt.round('1s') #round to one second for simplicity
        if data2.shape[0] < 86400: # if the number of datapoints is lower than one day:
            #print('Data gaps detected, padding array....')
            data3 = data2.rename(index=(data2['t_utc']-data2.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
            data3 = data3.reindex(range(0,86400)) # now we just fill in the missing values
            newt = pd.date_range(start = data2.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
            data3['t_utc'] = newt # now fill in the times so there is no NaT
            data2 = data3 
            del(data3)
        elif data2.shape[0] > 86400:
            error('Data file is too long, probably need to debug the code again....')
            print('Done\n')
        
        list_of_plasma.append(data2['npl'])    
        

['C:/Users/atell/OneDrive/Escritorio/ESA/trainwave/LAP DATA/LAP_20150210_000108_805_NEL.TAB']


#### Magnetic Fields Data

In [72]:
#folder with data

#folder="C:/Users/Ariel/Desktop/ESA/example1" #november 2014
#folder="C:/Users/Ariel/Desktop/ESA/example1" #december 2014

#folder="C:/Users/Ariel/Desktop/ESA/example1" #february 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #march 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #april 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #may 2015

folder="C:/Users/atell/OneDrive/Escritorio/ESA/trainwave/MAG DATA" #trainwave 2015
#folder="C:/Users/atell/OneDrive/Escritorio/esa compuviejo/examplee1" #singleevent 2015

#folder="C:/Users/Ariel/Desktop/ESA/example1" #july 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #august 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #september 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #october 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #november 2015
#folder="C:/Users/Ariel/Desktop/ESA/example1" #december 2015

#folder="C:/Users/Ariel/Desktop/ESA/example1" #january 2016
#folder="C:/Users/Ariel/Desktop/ESA/example1" #february 2016
#folder="C:/Users/Ariel/Desktop/ESA/example1" #march 2016

directory= os.scandir(folder)
#---------------------------------------------------------

### Reading Tables Function

In [74]:
#This function receives a path and returns the table of the MM Waves of that day and the magnetic fields/plasma density plots
def reading_table(path, data_plasma, G1, G2, THETA, PHI, B, n):
    
    #Importing data
    titles=['Dates_and_Hours', '?', 'x', 'y', 'z', 'Bx', 'By', 'Bz', 'flag']
    data= pd.read_table(str(path), header=None, names=titles, sep='\s+', parse_dates=['Dates_and_Hours'])
    #----------------------------------------
    
    path_time=pd.to_datetime(data['Dates_and_Hours'])                             
    #It is needed to obtain the year, month and day of an specific path
    q=path_time.dt.year
    qq=path_time.dt.month
    qqq=path_time.dt.day
    year=q[0]
    month=qq[0]
    day=qqq[0]
    
    #Filling the data gaps
    data.Dates_and_Hours = data.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
    if data.shape[0] < 86400: # if the number of datapoints is lower than one day:
        #print('Data gaps detected, padding array....')
        data2 = data.rename(index=(data['Dates_and_Hours']-data.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
        data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
        newt = pd.date_range(start = data.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
        data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
        data = data2 
        del(data2)
    elif data.shape[0] > 86400:
        error('Data file is too long, probably need to debug the code again....')
    #print('Done\n')
    
    #Index of data will be helpful later 
    data['Index'] = data.index
    path_time=pd.to_datetime(data['Dates_and_Hours']) 

    #-------------------------------------
    
    bmodulus=[]
    bx=data['Bx']
    by=data['By']
    bz=data['Bz']

    #Calculates the magnetic field modulus for each point
    for i in np.arange(0,len(bx)):
        Bpoint=(bx[i]**2+by[i]**2+bz[i]**2)**(1/2)
        bmodulus.append(Bpoint)
        
    #Transform a list into a data column
    df = pd.DataFrame({'col':bmodulus})
    magneticfieldcolumn=df['col']
    newdeltaB=newdeltas(magneticfieldcolumn)
    
    #print(newdeltaB)
    #----------------------------------------
    
    
    #MAGNETIC FIELDS ROLLING AVERAGES
    rolling1=bx.rolling(600,center=True).mean() #Rolling average of the x-field  
    rolling2=by.rolling(600,center=True).mean() #Rolling average of the y-field  
    rolling3=bz.rolling(600,center=True).mean() #Rolling average of the z-field 
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #Rolling average of the mfield modulus 
    
    #---------------------------------------------------
    XX = data[['Bx', 'By', 'Bz']]
    
    #Creating a list with the background magnetic field vectors
    bgmgvector=[]

    for i in np.arange(0,len(rolling1)):
        bgmgvector.append((rolling1[i],rolling2[i],rolling3[i]))
        
    #PCA
    final=function1(XX,n, newdeltaB, bgmgvector) 
    
    print('final')
    
    listofratio1=[]
    listofratio2=[]
    listofthetas=[] #These lists are used for the angles plotting
    listofphis=[]
    for i in np.arange(0,len(final)):
        listofratio1.append(final[i][0])
        listofratio2.append(final[i][1])
        listofthetas.append(final[i][2])
        listofphis.append(final[i][3])
    
    #---------------------------------------------------------------
    
    #Selection criteria
    
    finallistthetas=[] #These lists are used for the plot section
    finallistphis=[]
    finallistdeltas=[]
    for i in np.arange(0,len(final)):
        if final[i][0]==np.nan and final[i][1]==np.nan:
            finallistthetas.append(np.nan)
            finallistphis.append(np.nan)
            finallistdeltas.append(np.nan)
            
        elif final[i][0]>=G1 and final[i][1]<=G2 and final[i][2]>=THETA and final[i][3]<=PHI and final[i][4]>=B: #Goodness 1 and 2, theta, phi, deltaB criteria
            finallistthetas.append(final[i][2])
            finallistphis.append(final[i][3])
            finallistdeltas.append(final[i][4])
        else:
            finallistthetas.append(np.nan)
            finallistphis.append(np.nan)
            finallistdeltas.append(np.nan)
    
    #------------------------------------------------------------------   
    
    #BUILDING THE TABLE
    
    listofmgflds=[]
    #check_for_nan = finallistthetas.isnull()
    for i in np.arange(0,len(bmodulus)):
        if np.isnan(finallistthetas[i])==True:
            listofmgflds.append(np.nan)
        else:
            listofmgflds.append(bmodulus[i])
    
    listofindex=[]
    #check_for_nan = finallistthetas.isnull()
    for i in np.arange(0,len(finallistthetas)):
        if np.isnan(finallistthetas[i])!=True:
            listofindex.append(i)

    newdata=data.iloc[listofindex,:]
    newdataf=newdata[['Dates_and_Hours', 'Bx','By','Bz','Index']]
    #print(newdataf)
    tablegoodness1=[]
    tablegoodness2=[]
    tablethetas=[]
    tablephis=[]
    tablemaxdeltab=[]
    for i in listofindex:
        tablegoodness1.append(final[i][0])
        tablegoodness2.append(final[i][1])
        tablethetas.append(final[i][2])
        tablephis.append(final[i][3])
        tablemaxdeltab.append(final[i][4])
    #-------------------------------------------------------------------
    
    #TABLE
    d = {'Dates_and_Hours': newdataf['Dates_and_Hours'], 'Bx': newdataf['Bx'], 'By': newdataf['By'], 'Bz': newdataf['Bz'], 'Goodness1':tablegoodness1, 'Goodness2':tablegoodness2 ,'Thetas':tablethetas, 'Phis':tablephis, 'MaxDeltaB/B':tablemaxdeltab, 'Index': newdataf['Index'] }
    MMTABLE = pd.DataFrame(data=d)
    #-------------------------------------------------------------------
 
    #PLASMA DENSITY SECTION
    
    plasma_density_of_that_day=getting_day(data_plasma,year, month, day)
    
    if len(plasma_density_of_that_day[0])==0:
        #New dataframe for the plasma density 
        a1={'Dates_and_Hours':  [], 'npl': []}
        data_plasma_of_that_day= pd.DataFrame (a1, columns = ['Dates_and_Hours','npl'])
    else:
        
        #New dataframe for the plasma density 
        a1={'Dates_and_Hours':  plasma_density_of_that_day[0], 'npl': plasma_density_of_that_day[1]}
        data_plasma_of_that_day= pd.DataFrame (a1, columns = ['Dates_and_Hours','npl'])
    
        #Filling the data gaps
        data_plasma_of_that_day.Dates_and_Hours = data_plasma_of_that_day.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
        if data_plasma_of_that_day.shape[0] < 86400: # if the number of datapoints is lower than one day:
            #print('Data gaps detected, padding array....')
            data2 = data_plasma_of_that_day.rename(index=(data_plasma_of_that_day['Dates_and_Hours']-data_plasma_of_that_day.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
            data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
            newt = pd.date_range(start = data_plasma_of_that_day.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
            data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
            data_plasma_of_that_day = data2 
            del(data2)
        elif data_plasma_of_that_day.shape[0] > 86400:
            error('Data file is too long, probably need to debug the code again....')
    #----------------------------------------------------------------------------------------------------
    #New dataframe for the correlation method
    a2={'Dates_and_Hours': data['Dates_and_Hours'], 'bmodulus': bmodulus, 'rolling4': rolling4}
    data_correlation= pd.DataFrame(a2,columns = ['Dates_and_Hours', 'bmodulus', 'rolling4'])
    
    #print(MMTABLE)
    return MMTABLE, year, month, day, path_time, bmodulus,rolling1, rolling2, rolling3, rolling4, data_plasma_of_that_day['Dates_and_Hours'], data_plasma_of_that_day['npl'], listofmgflds, listofthetas, listofphis, data['Bx'], data['By'], data['Bz'], newdeltaB, data_correlation,     XX, newdeltaB, bgmgvector
   
    

### Iteration for every month

In [76]:
#IMPORTING ALL DATA FROM THE FOLDER

#First a folder with 2 files will be used in order to see if it works

list_of_files=get_directory(folder)

newlist=[]
for item in list_of_files:
    newlist.append(folder+'/'+str(item))
print(newlist)  

#------------------------------------
#ITERATION

#READING_TABLE PARAMETERS
#1) item: each element
#2) data_plasma: plasma density data
#3) G1: eigenvalue ratio1 criteria
#4) G2; eigenvalue ratio2 criteria
#5) THETA: theta criteria
#6) PHI: phi criteria
#7) B: deltaB criteria
#8) n: seconds used for the PCA method

#INSERT VALUES

#used in our study
G1=3
G2=6
THETA=70
PHI=20
B=1
n=30

#used by volwerk2016
#G1=0.1
#G2=15
#THETA=70
#PHI=20
#B=1
#n=30
#---------------------
MM_candidates=[]
MM_final_table=[]
count=[]
years=[]
months=[]
days=[]
#--------------------    
array1=[] #path_time
array2=[] #bmodulus
array3=[] #rolling1
array4=[] #rolling2
array5=[] #rolling3
array6=[] #rolling4
array7=[] #plasma_density_of_that_day[0]
array8=[] #plasma_density_of_that_day[1]
array9=[] #listofmgflds
array10=[] #listofthetas
array11=[] #listofphis
array12=[] #data['bx']
array13=[] #data['by']
array14=[] #data['bz']
array15=[] #newdeltaB

array16=[] #data_correlation
mm1=[]

#------------------------
for item in newlist:
    element=reading_table(item, data_plasma,G1,G2,THETA,PHI,B,n)
    MM_candidates.append(element[0])
    years.append(element[1])
    months.append(element[2])
    days.append(element[3])
    
    array1.append(element[4])
    array2.append(element[5])
    array3.append(element[6])
    array4.append(element[7])
    array5.append(element[8])
    array6.append(element[9])
    array7.append(element[10])
    array8.append(element[11])
    array9.append(element[12])
    array10.append(element[13])
    array11.append(element[14])
    array12.append(element[15])
    array13.append(element[16])
    array14.append(element[17])
    array15.append(element[18]) 
    array16.append(element[19])
    mm1.append(element[0])
    table=intervals(element[0], n, element[4])
    MM_final_table.append(table[0])
    count.append(table[1])
    
    #print(table[0])
    print('We have '+str(table[1])+'MM Waves')

['C:/Users/atell/OneDrive/Escritorio/ESA/trainwave/MAG DATA/RPCMAG150210_CLG_OB_A1.TAB']
final
We have 21MM Waves


In [94]:
#print(MM_candidates[0])
print(MM_final_table[0])

             Beginning                 End  Index1  Index2
0  2015-02-10 00:06:16 2015-02-10 00:07:08     391     413
1  2015-02-10 00:08:00 2015-02-10 00:08:51     495     516
2  2015-02-10 00:13:52 2015-02-10 00:14:53     847     878
3  2015-02-10 00:14:52 2015-02-10 00:15:45     907     930
4  2015-02-10 00:19:32 2015-02-10 00:20:04    1187    1189
5  2015-02-10 00:23:01 2015-02-10 00:23:46    1396    1411
6  2015-02-10 00:23:54 2015-02-10 00:24:40    1449    1465
7  2015-02-10 00:58:25 2015-02-10 00:59:12    3520    3537
8  2015-02-10 01:56:16 2015-02-10 01:56:46    6991    6991
9  2015-02-10 02:07:06 2015-02-10 02:07:52    7641    7657
10 2015-02-10 02:12:05 2015-02-10 02:12:36    7940    7941
11 2015-02-10 02:12:47 2015-02-10 02:13:24    7982    7989
12 2015-02-10 06:09:24 2015-02-10 06:09:57   22179   22182
13 2015-02-10 06:09:52 2015-02-10 06:10:26   22207   22211
14 2015-02-10 06:10:17 2015-02-10 06:10:53   22232   22238
15 2015-02-10 06:15:29 2015-02-10 06:15:59   22544   225

In [78]:
#Needed to fix plot bugs

for i in np.arange(0,len(array7)):
    if len(array7[i])==0:
        new_col = np.empty((array16[0].shape[0],1)) #f not it is filled with nan values
        new_col.fill(np.nan)
        new_col=new_col.astype(float)
        array7[i]=new_col
        
        new_col2= np.empty((array16[0].shape[0],1)) #f not it is filled with nan values
        new_col2.fill(np.nan)
        new_col2=new_col2.astype(float)
        array8[i]=new_col2

In [79]:
#done

#### Dataframe for the plots

In [82]:
#Dataframe with all data needed for the plots

finaldatas=[]
#I wont save the dates
for i in np.arange(0,len(array1)): #for every day
    year=years[i]*np.ones(len(array2[i]))
    year=year.tolist()
    month=months[i]*np.ones(len(array2[i]))
    month=month.tolist()
    day=days[i]*np.ones(len(array2[i]))
    day=day.tolist()
    if array8[i].shape==(len(array2[i]),1):
        z=array8[i].copy()
        x=z.tolist() 
    else:
        x=array8[i].copy()
    name={ 'array2': array2[i],'array3': array3[i],'array4': array4[i],'array5': array5[i],'array6': array6[i],'array8': x,'array9': array9[i],'array10': array10[i],'array11': array11[i],'array12': array12[i],'array13': array13[i],'array14': array14[i],'array15': array15[i], 'year':year,'month':month,'day':day }
    alldata= pd.DataFrame(name)#,columns = ['array1', 'bmodulus'])

    
    finaldatas.append(alldata)


### Anti Correlation

In [84]:
#First of all it is needed to change the frequency of data in order to make the anticorrelation

In [100]:
listt=[] #List with the anticorrelated candidates
list2=[] #list with all candidates with their respective pearson value

final_events=[] #list with dataframes (bmodulus, npl)
final_events_plots=[] #list with larger dataframes in order to have a better plot
for i in np.arange(0,len(MM_final_table)): #for every day
    
    if len(MM_final_table[i])==0: #If there are not MM mode candidates
        listt.append(pd.DataFrame([]))
        list2.append(pd.DataFrame([]))
        final_events.append(pd.DataFrame([]))
        
    else:    
        #PARA UN SOLO DIA
        array16copy=array16[i].copy()
        path_time=pd.to_datetime(array16copy['Dates_and_Hours'])

        #Creating a new dataframe in order to establish the anticorrelation
        a2={ 'Dates_and_Hours': array16copy['Dates_and_Hours'], 'bmodulus': array16copy['bmodulus'], 'rolling4': array16copy['rolling4']}
        data_correlation2= pd.DataFrame(a2,columns = ['Dates_and_Hours', 'bmodulus','rolling4'])
        data_correlation2['index']=path_time

        
        #-------------------------------------------------
        #Finding the correct npl list for the day
        if days[i] in day_plasma:
            indexx=[]
            for k in np.arange(0,len(day_plasma)):
                if days[i]==day_plasma[k]:
                    indexx.append(k)
            
            LAP_plasma=list_of_plasma[indexx[0]]
        else:
            LAP_plasma=[]
        #-------------------------------------------------- 
        
        
        #-----------------------------------------------------------------------------------
        #Adding the plasma density to the dataframe
        if len(LAP_plasma)!=0: #If we have high density plasma data
            data_correlation2['npl']=LAP_plasma
        else:
            new_col = np.empty((array16[i].shape[0],1)) #if not it is filled with nan values
            new_col.fill(np.nan)
            data_correlation2['npl']=new_col

        data_correlation2.set_index('index', inplace=True)  
       
        
        
        
        
        #-----------------------------------------------------------------------------------

        #-----------------------------------------------------------------------------------    
        #ITERATION FOR EVERY EVENT
        
       
        events_to_correlate=[] #List of dataframes that includes bmodulus and npl for every time interval WITHOUT RESAMPLE
        events_to_correlate2=[] #WITH RESAMPLE 
        
        events_to_correlate_plot=[]
        #print(MM_final_table[i])
        for j in np.arange(0,len(MM_final_table[i])):
            #print(j)
            start=str(MM_final_table[i]['Beginning'][j])
            start_date = datetime.datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
            start_date2=start_date.strftime('%H:%M:%S') #Time with the format needed
            end=str(MM_final_table[i]['End'][j])
            end_date=datetime.datetime.strptime(end, '%Y-%m-%d %H:%M:%S')
            end_date2=end_date.strftime('%H:%M:%S') #Time with the format needed
            #Choosing the time
            df1 = data_correlation2.between_time(start_time=start_date2, end_time=end_date2) #Dataframe with the MM event data
            
            
            #for the plot
            startplot=str(MM_final_table[i]['Beginning'][j])
            startplot_date=datetime.datetime.strptime(startplot, '%Y-%m-%d %H:%M:%S')-pd.Timedelta(minutes=1)
            startplot_date2=startplot_date.strftime('%H:%M:%S') #Time with the format needed
            endplot=str(MM_final_table[i]['End'][j])
            endplot_date=datetime.datetime.strptime(endplot, '%Y-%m-%d %H:%M:%S')+pd.Timedelta(minutes=1)
            endplot_date2=endplot_date.strftime('%H:%M:%S') #Time with the format needed
            dfplot = data_correlation2.between_time(start_time=startplot_date2, end_time=endplot_date2) #Dataframe with the MM event data
            
            dfplot2=dfplot.copy()
            dfplot2['index'] = pd.to_datetime(data_correlation2['Dates_and_Hours'])
            dfplot2.set_index('index', inplace=True)
            dfplot2=dfplot2.resample('3S').mean(numeric_only=True)
            dfplot2['t_utc'] = pd.to_datetime(dfplot2.index.values)
            dfplot2= dfplot2.reset_index()   
        
        
            #---------------------
            #Resample

            df2=df1.copy()
            df2['index'] = pd.to_datetime(data_correlation2['Dates_and_Hours'])
            df2.set_index('index', inplace=True)
            df2=df2.resample('3S').mean(numeric_only=True)
            df2['t_utc'] = pd.to_datetime(df2.index.values)
            df2= df2.reset_index()
            #---------------------
            #print(df1)
            #print('------------------')
            #print(df2)

            events_to_correlate.append(df1)
            events_to_correlate2.append(df2)
            events_to_correlate_plot.append(dfplot2)
        final_events.append(events_to_correlate2)
        final_events_plots.append(events_to_correlate_plot)
        
        
        pearson=[] #List of pearson dataframes for every event
        pearson2=[]
        #CORRELATION FOR EVERY EVENT
        for j in np.arange(0,len(events_to_correlate2)):
            sss=events_to_correlate[j].corr(method='pearson',numeric_only=True)
            pearson.append(sss)
            eee=events_to_correlate2[j].corr(method='pearson',numeric_only=True)
            pearson2.append(eee)
        #-------------------------------------------------    

        #New dataframes
        
        #Dataframe of all events with pearson values
        MM_final_table_with_pearson=MM_final_table[i].copy()
        pearson_list=[]
        for j in np.arange(0,len(pearson2)):
            pearson_list.append(pearson2[j]['bmodulus'][2])
        MM_final_table_with_pearson['Pearson']=pearson_list
        list2.append(MM_final_table_with_pearson)    
        
        #Dataframe only with anticorrelated values
        MM_final_table_copy=MM_final_table_with_pearson.copy() 
        for j in np.arange(0,len(pearson2)):
            if pearson2[j]['bmodulus'][2]>-0.7 or np.isnan(pearson2[j]['bmodulus'][2])==True: #if the interval is not anti correlated or there is not high density plasma dat
                MM_final_table_copy['Beginning'][j]=np.nan
                MM_final_table_copy['End'][j]=np.nan
        
        for j in np.arange(0,len(MM_final_table_copy)):
            if pd.isnull(MM_final_table_copy['Beginning'][j])==True and pd.isnull(MM_final_table_copy['End'][j])==True:
                MM_final_table_copy.drop(j, axis=0, inplace=True)
        
        MM_final_table_copy=MM_final_table_copy.reset_index(drop=True)
        #--------------------------------------------------
        
        if len(MM_final_table_copy)==0:
            listt.append(pd.DataFrame([]))
        else:    
            listt.append(MM_final_table_copy)

C:\Users\atell\AppData\Local\Temp\ipykernel_23332\874722895.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  MM_final_table_copy['Beginning'][j]=np.nan
C:\Users\atell\AppData\Local\Temp\ipykernel_23332\874722895.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  MM_final_table_copy['End'][j]=np.nan


In [86]:
print(MM_final_table)
print(listt)
print(list2)

[             Beginning                 End  Index1  Index2
0  2015-02-10 00:06:16 2015-02-10 00:07:08     391     413
1  2015-02-10 00:08:00 2015-02-10 00:08:51     495     516
2  2015-02-10 00:13:52 2015-02-10 00:14:53     847     878
3  2015-02-10 00:14:52 2015-02-10 00:15:45     907     930
4  2015-02-10 00:19:32 2015-02-10 00:20:04    1187    1189
5  2015-02-10 00:23:01 2015-02-10 00:23:46    1396    1411
6  2015-02-10 00:23:54 2015-02-10 00:24:40    1449    1465
7  2015-02-10 00:58:25 2015-02-10 00:59:12    3520    3537
8  2015-02-10 01:56:16 2015-02-10 01:56:46    6991    6991
9  2015-02-10 02:07:06 2015-02-10 02:07:52    7641    7657
10 2015-02-10 02:12:05 2015-02-10 02:12:36    7940    7941
11 2015-02-10 02:12:47 2015-02-10 02:13:24    7982    7989
12 2015-02-10 06:09:24 2015-02-10 06:09:57   22179   22182
13 2015-02-10 06:09:52 2015-02-10 06:10:26   22207   22211
14 2015-02-10 06:10:17 2015-02-10 06:10:53   22232   22238
15 2015-02-10 06:15:29 2015-02-10 06:15:59   22544   22

### Anti Correlation plot

In [102]:
from pandas import DataFrame
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


ysize=10
xsize=10
minutes=15


#for i in np.arange(0,len(MM_final_table)):
for i in np.arange(0,1):
    #with PdfPages(r'C:/Users/Ariel/Desktop/JUPYTER/'+'Plots_anti_correlation'+str(years[i])+'_'+str(months[i])+'_'+str(days[i])+'.pdf') as export_pdf:
    with PdfPages(r'C:/Users/atell/OneDrive/Escritorio/ESA/plots/'+'Plots_anti_correlation'+str(years[i])+'_'+str(months[i])+'_'+str(days[i])+'.pdf') as export_pdf:
        #for j in np.arange(78,79):
        #for j in np.arange(20,26):
        for j in np.arange(2,3):        
            
            
#        for j in np.arange(0,len(final_events[i])):
            
            if len(list2[i])!=0:
            
                if pd.isnull(list2[i]['Pearson'][j])==False:
                
                    #Plasma density
                    plt.figure()
                    ax7=plt.subplot(211)
                    figsize=(20,4)
                    #plt.plot(final_events[i][j]['t_utc'], final_events[i][j]['npl'], '-.',c='black', label='npl') 
                    plt.plot(final_events_plots[i][j]['t_utc'], final_events_plots[i][j]['npl'], '-.',c='black', label='npl') 
                    plt.ylabel('n ($cm^{-3}$)',fontsize=10)        
                    ax7.tick_params(axis="y", labelsize=ysize)
                    ax7.yaxis.set_label_position("right")
                    ax7.yaxis.tick_right()
                    #plt.xlabel('Time (UTC)',fontsize=10)
                    xfmt = mdates.DateFormatter("%H:%M:%S")
                    ax7.xaxis.set_major_formatter(xfmt)
                    #plt.setp(ax7.get_xticklabels(), fontsize=6)
                    plt.setp(ax7.get_xticklabels(), visible=False)
                    #plt.legend()
                    plt.title('Pearson='+str(list2[i]['Pearson'][j]),fontsize=15)
                    ax7.get_yaxis().set_label_coords(1.092,0.5)
                    
                    plt.axvline(final_events[i][j]['t_utc'][0], c='purple') 
                    plt.axvline(final_events[i][j]['t_utc'][len(final_events[i][j])-1], c='purple')
                    print('inicio', final_events[i][j]['t_utc'][0])
                    print('final', final_events[i][j]['t_utc'][len(final_events[i][j])-1])

                    #bmodulus
                    ax2=plt.subplot(212, sharex=ax7)
                    figsize=(20,4)
                    #plt.plot(final_events[i][j]['t_utc'], final_events[i][j]['bmodulus'], '-.',c='black', label='bmodulus')
                    plt.plot(final_events_plots[i][j]['t_utc'], final_events_plots[i][j]['bmodulus'], '-.',c='black', label='bmodulus')
                    plt.plot(final_events_plots[i][j]['t_utc'], final_events_plots[i][j]['rolling4'], '-.',c='red', label='rolling') 
                    plt.ylabel('|B| (nT)',fontsize=10)

                    ax2.tick_params(axis="y", labelsize=ysize)
                    ax2.yaxis.set_label_position("right")
                    ax2.yaxis.tick_right()
                    plt.xlabel('Time (UTC)',fontsize=10)
                    xfmt = mdates.DateFormatter("%H:%M:%S")
                    ax2.xaxis.set_major_formatter(xfmt)
                    plt.setp(ax2.get_xticklabels(), fontsize=xsize)
                    ax2.get_yaxis().set_label_coords(1.092,0.5)
                    
                    plt.axvline(final_events[i][j]['t_utc'][0], c='purple') 
                    plt.axvline(final_events[i][j]['t_utc'][len(final_events[i][j])-1], c='purple')

                    #plt.legend()

                    export_pdf.savefig()
                    #plt.close()

inicio 2015-02-10 00:13:51
final 2015-02-10 00:14:51


In [104]:
from pandas import DataFrame
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


ysize=10
xsize=10
minutes=15


#for i in np.arange(0,len(MM_final_table)):
for i in np.arange(0,1):
    #with PdfPages(r'C:/Users/Ariel/Desktop/JUPYTER/'+'Plots_anti_correlation'+str(years[i])+'_'+str(months[i])+'_'+str(days[i])+'.pdf') as export_pdf:
    with PdfPages(r'C:/Users/atell/OneDrive/Escritorio/ESA/plots/'+'Plots_anti_correlation'+str(years[i])+'_'+str(months[i])+'_'+str(days[i])+'.pdf') as export_pdf:
        #for j in np.arange(78,79):
        #for j in np.arange(20,26):
        for j in np.arange(2,3):        
            
            
#        for j in np.arange(0,len(final_events[i])):
            
            if len(list2[i])!=0:
            
                if pd.isnull(list2[i]['Pearson'][j])==False:
                
                    #Plasma density
                    plt.figure()
                    ax7=plt.subplot(211)
                    figsize=(20,4)
                    #plt.plot(final_events[i][j]['t_utc'], final_events[i][j]['npl'], '-.',c='black', label='npl') 
                    plt.plot(final_events[i][j]['t_utc'], final_events[i][j]['npl'], 'x',c='black', label='npl') 
                    plt.ylabel('n ($cm^{-3}$)',fontsize=10)        
                    ax7.tick_params(axis="y", labelsize=ysize)
                    ax7.yaxis.set_label_position("right")
                    ax7.yaxis.tick_right()
                    #plt.xlabel('Time (UTC)',fontsize=10)
                    xfmt = mdates.DateFormatter("%H:%M:%S")
                    ax7.xaxis.set_major_formatter(xfmt)
                    #plt.setp(ax7.get_xticklabels(), fontsize=6)
                    plt.setp(ax7.get_xticklabels(), visible=False)
                    #plt.legend()
                    plt.title('Pearson='+str(list2[i]['Pearson'][j]),fontsize=15)
                    ax7.get_yaxis().set_label_coords(1.092,0.5)
                    
                    plt.axvline(final_events[i][j]['t_utc'][0], c='purple') 
                    plt.axvline(final_events[i][j]['t_utc'][len(final_events[i][j])-1], c='purple')
                    print('inicio', final_events[i][j]['t_utc'][0])
                    print('final', final_events[i][j]['t_utc'][len(final_events[i][j])-1])

                    #bmodulus
                    ax2=plt.subplot(212, sharex=ax7)
                    figsize=(20,4)
                    #plt.plot(final_events[i][j]['t_utc'], final_events[i][j]['bmodulus'], '-.',c='black', label='bmodulus')
                    plt.plot(final_events[i][j]['t_utc'], final_events[i][j]['bmodulus'], 'x',c='black', label='bmodulus')
                    plt.plot(final_events[i][j]['t_utc'], final_events[i][j]['rolling4'], '-.',c='red', label='rolling') 
                    plt.ylabel('|B| (nT)',fontsize=10)

                    ax2.tick_params(axis="y", labelsize=ysize)
                    ax2.yaxis.set_label_position("right")
                    ax2.yaxis.tick_right()
                    plt.xlabel('Time (UTC)',fontsize=10)
                    xfmt = mdates.DateFormatter("%H:%M:%S")
                    ax2.xaxis.set_major_formatter(xfmt)
                    plt.setp(ax2.get_xticklabels(), fontsize=xsize)
                    ax2.get_yaxis().set_label_coords(1.092,0.5)
                    
                    plt.axvline(final_events[i][j]['t_utc'][0], c='purple') 
                    plt.axvline(final_events[i][j]['t_utc'][len(final_events[i][j])-1], c='purple')

                    #plt.legend()

                    export_pdf.savefig()
                    #plt.close()

inicio 2015-02-10 00:13:51
final 2015-02-10 00:14:51


In [106]:
print(final_events[0][2]['t_utc'])
print(final_events[0][2]['npl'])
print(final_events[0][2]['bmodulus'])

# Crear un diccionario que contenga las listas como valores
data = {'Columna1': final_events[0][2]['npl'] ,'Columna2': final_events[0][2]['bmodulus']}

# Crear el DataFrame
df = pd.DataFrame(data)

# Imprimir el DataFrame resultante
print(df)

sess=df.corr(method='pearson')
print(sess)

0    2015-02-10 00:13:51
1    2015-02-10 00:13:54
2    2015-02-10 00:13:57
3    2015-02-10 00:14:00
4    2015-02-10 00:14:03
5    2015-02-10 00:14:06
6    2015-02-10 00:14:09
7    2015-02-10 00:14:12
8    2015-02-10 00:14:15
9    2015-02-10 00:14:18
10   2015-02-10 00:14:21
11   2015-02-10 00:14:24
12   2015-02-10 00:14:27
13   2015-02-10 00:14:30
14   2015-02-10 00:14:33
15   2015-02-10 00:14:36
16   2015-02-10 00:14:39
17   2015-02-10 00:14:42
18   2015-02-10 00:14:45
19   2015-02-10 00:14:48
20   2015-02-10 00:14:51
Name: t_utc, dtype: datetime64[ns]
0     149.662870
1     214.701690
2     445.948133
3     323.095875
4     409.813060
5     185.790780
6     142.431321
7      84.619586
8       5.128986
9     243.599018
10    724.150610
11    395.350650
12     62.939466
13     19.579615
14     41.258299
15    135.199020
16     99.067682
17     91.841110
18    377.273445
19    200.232497
20    286.944847
Name: npl, dtype: float64
0     19.196055
1     16.532443
2     11.323462
3     10.

### Saving files

In [110]:
import csv

#SAVING THE TABLES IN A CSV FILE
for i in np.arange(0,len(MM_final_table)):
    MM_candidates[i].pop('tvalue')
    MM_candidates[i].pop('delta')
    MM_candidates[i].to_csv('MM_candidates'+str(years[i])+'_'+str(months[i])+'_'+str(days[i]), index=False, sep='\t')
    #MM_final_table[i].to_csv('MM_final_table'+'_'+str(years[i])+'_'+str(months[i])+'_'+str(days[i]), index=False, sep='\t')
    list2[i].to_csv('MM_final_table_with_pearson'+'_'+str(years[i])+'_'+str(months[i])+'_'+str(days[i]), index=False, sep='\t')
    listt[i].to_csv('MM_final_table_anticorrelation'+'_'+str(years[i])+'_'+str(months[i])+'_'+str(days[i]), index=False, sep='\t')
    finaldatas[i].to_csv('DATA'+'_'+str(years[i])+'_'+str(months[i])+'_'+str(days[i]), index=False, sep='\t')


KeyError: 'tvalue'

### ----------------

### TEST

In [ ]:
#angles plot 1
ax7=plt.subplot(7,1,7)
figsize=(10,4)
#fig.autofmt_xdate()
plt.plot(array1[0], array10[0], '.',c='green', label='\u03B8') 
plt.plot(array1[0], array11[0], '.',c='red', label='\u03C6') 
plt.plot(array1[0], THETA*np.ones(len(array1[0])),c='black')
plt.plot(array1[0], PHI*np.ones(len(array1[0])),c='black') 
plt.ylabel('\u03B8 , \u03C6 (deg)',fontsize=5)
ax7.tick_params(axis="y", labelsize=5)
plt.xlabel('Time (UTC)',fontsize=10)
ax7.set_yticks([0, 90, 180])
ax7.set_ylim([-5,(max(np.nanmax(array10[0]), np.nanmax(array11[0]))+10)]) #-5 and 10
xfmt = mdates.DateFormatter("%H:%M")
ax7.xaxis.set_major_formatter(xfmt)
plt.setp(ax7.get_xticklabels(), fontsize=6)
ax7.set(xlim=[array1[0][(MM_final_table[0]['Index1'][len(MM_final_table[0])-1])-300],array1[0][(MM_final_table[0]['Index2'][len(MM_final_table[0])-1])+300]]) #plot interval of 10 minutes
#ax7.set(xlim=[array1[0][(MM_final_table[0]['Index1'][0]-300)],array1[0][(MM_final_table[0]['Index2'][0]+300)]]) #plot interval of 10 minutes

ax7.xaxis.set_major_locator(mdates.MinuteLocator(interval=3))

plt.axvline(array1[0][MM_final_table[0]['Index1'][len(MM_final_table[0])-1]-30], color='purple')
plt.axvline(array1[0][MM_final_table[0]['Index2'][len(MM_final_table[0])-1]+30], color='purple')

                
#Magnetic field in the x direction 1
ax1=plt.subplot(7,1,1, sharex=ax7)
figsize=(10,4)
#fig.autofmt_xdate()
plt.plot(array1[0], array12[0],c='black') 
plt.plot(array1[0],array3[0],"r-") 
                
v1=[]
ind1=np.arange((MM_final_table[0]['Index1'][len(MM_final_table[0])-1])-300,(MM_final_table[0]['Index2'][len(MM_final_table[0])-1])+300+1) #list with index 
for item in ind1:
    v1.append(array12[0][item])
                    
ax1.set_ylim([-5+np.nanmin(v1),np.nanmax(v1)+5]) #-5 and 5
plt.ylabel('$B_x$ (nT)',fontsize=5)
ax1.tick_params(axis="y", labelsize=5)
xfmt = mdates.DateFormatter("%H:%M")
ax1.xaxis.set_major_formatter(xfmt)
plt.setp(ax1.get_xticklabels(), visible=False)
ax1.xaxis.set_major_locator(mdates.MinuteLocator(interval=3))

plt.axvline(array1[0][MM_final_table[0]['Index1'][len(MM_final_table[0])-1]-30], color='purple')
plt.axvline(array1[0][MM_final_table[0]['Index2'][len(MM_final_table[0])-1]+30], color='purple')


#High resolution plasma density

ax2=plt.subplot(7,1,2, sharex=ax7)
figsize=(10,4)
#fig.autofmt_xdate()
time_lap=pd.to_datetime(data2['t_utc'])
plt.plot(time_lap, data2['npl'],'b-') #,c='blue')  
                
v2=[]
for item in ind1:
    v2.append(data2['npl'][item])
                    
ax2.set_ylim([-5+np.nanmin(v2),np.nanmax(v2)+5]) #-5 and 5
plt.ylabel('$B_x$ (nT)',fontsize=5)
ax1.tick_params(axis="y", labelsize=5)
xfmt = mdates.DateFormatter("%H:%M")
ax1.xaxis.set_major_formatter(xfmt)
plt.setp(ax1.get_xticklabels(), visible=False)
ax1.xaxis.set_major_locator(mdates.MinuteLocator(interval=3))

plt.axvline(array1[0][MM_final_table[0]['Index1'][len(MM_final_table[0])-1]-30], color='purple')
plt.axvline(array1[0][MM_final_table[0]['Index2'][len(MM_final_table[0])-1]+30], color='purple')

## ?